# Annuaire Statistique du Maroc 2024 (HCP)

## Population dataset

In [364]:
import numpy as np
import pandas as pd

path="../data_processed/"

population_par_region_df = pd.read_csv(path + '01_population_par_region.csv')


population_par_region_df.head()

,Region,Annee,Milieu,Population
0,Tanger - Tétouan - Al Hoceima,2023,Total,3942576
1,Tanger - Tétouan - Al Hoceima,2022,Total,3900359
2,Tanger - Tétouan - Al Hoceima,2023,Urbain,2490225
3,Tanger - Tétouan - Al Hoceima,2022,Urbain,2450383
4,Tanger - Tétouan - Al Hoceima,2023,Rural,1452351


In [365]:
population_par_region_df.shape

(76, 4)

#### Vérification de manque de lignes

Préparer les valeurs uniques

In [366]:
regions = population_par_region_df['Region'].unique()
annees = population_par_region_df['Annee'].unique()
milieux = population_par_region_df['Milieu'].unique()

Créer toutes les combinaisons possibles

In [367]:
import itertools

# toutes les combinaisons possibles de Region x Annee x Milieu
all_combinations = pd.DataFrame(list(itertools.product(regions, annees, milieux)),
                                columns=['Region', 'Annee', 'Milieu'])

Trouver les lignes manquantes

In [368]:
# merge avec DataFrame pour trouver ce qui manque
merged = all_combinations.merge(population_par_region_df, 
                                on=['Region', 'Annee', 'Milieu'], 
                                how='left', 
                                indicator=True)

# garder uniquement les combinaisons qui ne sont pas dans le df
missing = merged[merged['_merge'] == 'left_only'][['Region', 'Annee', 'Milieu']]

Quand tu fais un merge avec indicator=True, pandas crée une colonne spéciale _merge qui indique d’où vient chaque ligne :

    -'both' → la combinaison est dans les deux DataFrames
    -'left_only' → la combinaison est seulement dans le DataFrame de gauche (ici all_combinations)
    -'right_only' → la combinaison est seulement dans le DataFrame de droite (ici population_par_region_df)
    
Donc merged['_merge'] == 'left_only' sélectionne toutes les combinaisons possibles qui n’existent pas dans ton DataFrame réel → ce sont les données manquantes.

In [369]:
print(missing)

Empty DataFrame
Columns: [Region, Annee, Milieu]
Index: []


#### Vérification des nulls

In [370]:
population_par_region_df.isnull().sum()

Region        0
Annee         0
Milieu        0
Population    0
dtype: int64

#### Vérification des duplicats

In [371]:
population_par_region_df.duplicated(['Region', 'Annee', 'Milieu']).sum()

4

In [372]:
population_par_region_df[population_par_region_df.duplicated(['Region', 'Annee', 'Milieu'], keep=False)]

,Region,Annee,Milieu,Population
68,Dakhla - Oued Ed-Dahab,2023,Urbain,166049
69,Dakhla - Oued Ed-Dahab,2022,Urbain,158104
70,Dakhla - Oued Ed-Dahab,2023,Rural,34798
71,Dakhla - Oued Ed-Dahab,2022,Rural,35023
72,Dakhla - Oued Ed-Dahab,2023,Urbain,23991045
73,Dakhla - Oued Ed-Dahab,2022,Urbain,23591583
74,Dakhla - Oued Ed-Dahab,2023,Rural,13031340
75,Dakhla - Oued Ed-Dahab,2022,Rural,13078633


alors après consulter le fichier excel qui concerne la population j'ai gardé les 4 premiers valeurs car les quatres valeurs derniers concerne l'ensemble des regions

In [373]:
population_par_region_df = population_par_region_df[
    population_par_region_df['Population'] < 10000000
]

#### supprimer toutes les lignes où Milieu == "Total"

In [374]:
population_par_region_df = population_par_region_df[population_par_region_df['Milieu'] != 'Total']

In [375]:
population_par_region_df.shape

(48, 4)

## Energie dataset

In [376]:
energie_par_region_df = pd.read_csv(path + '02_energie_ventes_electricite_par_province.csv')


energie_par_region_df.head()

,Region,Province_Prefecture,Annee,Ventes_Electricite_Mm_KWh
0,Tanger - Tétouan - Al Hoceima,Tanger - Tétouan - Al Hoceima,2023,4027.4006
1,Tanger - Tétouan - Al Hoceima,Tanger - Tétouan - Al Hoceima,2022,4005.2577
2,Tanger - Tétouan - Al Hoceima,Al Hoceima,2023,214.8958
3,Tanger - Tétouan - Al Hoceima,Al Hoceima,2022,206.7598
4,Tanger - Tétouan - Al Hoceima,Chefchaouen,2023,103.7691


In [377]:
energie_par_region_df.shape

(174, 4)

#### Garder Ventes_Electricite_Mm_KWh par region et pas par province

In [378]:
energie_par_region_df = energie_par_region_df[energie_par_region_df['Region'] == energie_par_region_df['Province_Prefecture']]

In [379]:
energie_par_region_df.drop(columns=['Province_Prefecture'], inplace=True)

In [380]:
energie_par_region_df.shape

(18, 3)

#### Vérification de manque de lignes

In [381]:
energie_par_region_df['Annee'].unique()

array([2023, 2022], dtype=int64)

In [382]:
energie_par_region_df['Region'].unique()

array(['Tanger - Tétouan - Al Hoceima', "L'Oriental", 'Fès - Meknès',
       'Rabat - Salé - Kénitra', 'Marrakech - Safi', 'Souss - Massa',
       'Guelmim - Oued Noun', 'Laâyoune - Sakia El Hamra',
       'Dakhla - Oued Ed-Dahab'], dtype=object)

on a pas des lignes manquantes car on a 9 regions et 18 lignes dans on a les vals des années 2022 et 2023

#### Vérification des nulls

In [383]:
energie_par_region_df.isnull().sum()

Region                       0
Annee                        0
Ventes_Electricite_Mm_KWh    0
dtype: int64

#### Vérification des duplicats

In [384]:
energie_par_region_df.duplicated(['Region', 'Annee']).sum()

0

## Eau production dataset

In [385]:
eau_production_par_centre_df = pd.read_csv(path + '03a_eau_production_superficielle_par_centre.csv')


eau_production_par_centre_df.head()

,Region,Province,Centre,Barrage_Oued,Annee,Production_Eau_m3
0,Tanger - Tétouan - Al Hoceima,Al Hociema,Al Hociema,Barrage Med Ben Abdelkrim Khettabi,2023,3050099.0
1,Tanger - Tétouan - Al Hoceima,Al Hociema,Al Hociema,Barrage Med Ben Abdelkrim Khettabi,2022,4858397.0
2,Tanger - Tétouan - Al Hoceima,NaN,Targuist,Barrage Joumoua,2023,638019.0
3,Tanger - Tétouan - Al Hoceima,NaN,Targuist,Barrage Joumoua,2022,343363.0
4,Tanger - Tétouan - Al Hoceima,Larache,Larache et Ksar El Kebir,Barrage Oued El Makhazine,2023,13439192.0


In [386]:
eau_production_par_centre_df.columns

Index(['Region', 'Province', 'Centre', 'Barrage_Oued', 'Annee',
       'Production_Eau_m3'],
      dtype='object')

#### Grouper par région

In [387]:
eau_production_par_centre_df.groupby(['Region', 'Annee'])['Production_Eau_m3'].sum()

Region                         Annee
Béni Mellal - Khénifra         2022     73574513.0
                               2023     76159151.0
Casablanca - Settat            2022      1265525.0
                               2023      1022558.0
Drâa - Tafilalet               2022      3644186.0
                               2023      3383267.0
Fès - Meknès                   2022     41393344.0
                               2023     33290883.0
L'Oriental                     2022     55688800.0
                               2023     58747097.0
Marrakech - Safi               2022     44502940.0
                               2023     47093716.0
Rabat - Salé - Kénitra         2022     19509375.0
                               2023     20536351.0
Souss - Massa                  2022     25980960.0
                               2023     17273486.0
Tanger - Tétouan - Al Hoceima  2022     30749668.0
                               2023     34621361.0
Name: Production_Eau_m3, dtype: float64

In [388]:
eau_production_par_region_df = eau_production_par_centre_df.groupby(
    ['Region', 'Annee'], as_index=False
)['Production_Eau_m3'].sum()

In [389]:
eau_production_par_region_df = eau_production_par_region_df.rename(
    columns={'Production_Eau_m3': 'Production_Eau_totale_m3'}
)

In [390]:
eau_production_par_region_df.head()

,Region,Annee,Production_Eau_totale_m3
0,Béni Mellal - Khénifra,2022,73574513.0
1,Béni Mellal - Khénifra,2023,76159151.0
2,Casablanca - Settat,2022,1265525.0
3,Casablanca - Settat,2023,1022558.0
4,Drâa - Tafilalet,2022,3644186.0


In [391]:
eau_production_par_region_df.shape

(18, 3)

#### Vérification des nulls

In [392]:
eau_production_par_region_df.isnull().sum()

Region                      0
Annee                       0
Production_Eau_totale_m3    0
dtype: int64

## PIB par secteur dataset

In [393]:
pib_par_region_df = pd.read_csv(path + '04_pib_regional_par_secteur.csv')

pib_par_region_df.head()

,Region,Annee,A00_Agriculture,A05_Peche,B00_Extraction,C00_Industrie_Manuf,DE0_Elec_Eau,F00_Construction,G00_Commerce,H00_Transport,I00_Hebergement,J00_InfoComm,K00_Finance,L68_Immobilier,MN0_Recherche,O84_AdminPublique,PQ8_Education,RS0_Autres,IS_Pt_Impots_Nets,PIB_Regional_MDH
0,Tanger - Tétouan - Al Hoceima,2022,12107.0,471.0,0.0,37724.0,1544.0,7342.0,14143.0,4787.0,4514.0,253.0,2201.0,10639.0,7013.0,9522.0,11009.0,1761.0,13984.0,139014.0
1,L'Oriental,2022,10742.0,71.0,728.0,2594.0,1908.0,7824.0,12756.0,1769.0,1592.0,209.0,1278.0,4161.0,3220.0,8336.0,5071.0,339.0,5819.0,68417.0
2,Fès - Meknès,2022,22828.0,0.0,24.0,6185.0,888.0,5824.0,16029.0,3357.0,3312.0,331.0,2869.0,7092.0,4677.0,10359.0,12640.0,1716.0,7916.0,105747.0
3,Rabat - Salé - Kénitra,2022,21465.0,56.0,87.0,17039.0,6553.0,11127.0,19397.0,6944.0,4233.0,16592.0,10512.0,15920.0,10808.0,37045.0,14682.0,3034.0,18511.0,214006.0
4,Béni Mellal - Khénifra,2022,11515.0,0.0,21281.0,1376.0,545.0,4715.0,8537.0,2583.0,1495.0,214.0,658.0,3664.0,3896.0,6094.0,6698.0,983.0,7560.0,81814.0


In [394]:
pib_par_region_df.shape

(24, 20)

#### Vérification des années concernées

In [395]:
pib_par_region_df["Annee"].unique()

array([2022, 2021], dtype=int64)

In [396]:
pib_par_region_2022_df = pib_par_region_df[pib_par_region_df["Annee"] == 2022].copy()

In [397]:
pib_par_region_2022_df["Annee"].unique()

array([2022], dtype=int64)

#### Vérification des duplicats

In [398]:
pib_par_region_2022_df.shape

(12, 20)

In [399]:
pib_par_region_2022_df["Region"].unique()

array(['Tanger - Tétouan - Al Hoceima', "L'Oriental", 'Fès - Meknès',
       'Rabat - Salé - Kénitra', 'Béni Mellal - Khénifra',
       'Casablanca - Settat', 'Marrakech - Safi', 'Drâa - Tafilalet',
       'Souss - Massa', 'Guelmim - Oued Noun',
       'Laâyoune - Sakia El Hamra', 'Dakhla - Oued Ed-Dahab'],
      dtype=object)

In [400]:
pib_par_region_2022_df.duplicated('Region').sum()

0

## Transport routes dataset

In [401]:
transport_par_region_2023_df= pd.read_csv(path + '07_transport_routes_par_region_2023.csv')

transport_par_region_2023_df.head()

,Region,Annee,Routes_Provinciales_Total_km,Routes_Provinciales_Revetues_km,Routes_Regionales_Total_km,Routes_Nationales_Total_km
0,Béni Mellal - Khénifra,2023,2433.893,1948.658,945.461,938.429
1,Casablanca - Settat,2023,3885.049,3306.585,1200.039,796.271
2,Drâa - Tafilalet,2023,2224.300,1425.595,1331.270,1841.570
3,Fès - Meknès,2023,4665.622,3569.435,1363.993,1413.105
4,Guelmim - Oued Noun,2023,1619.214,976.714,595.616,785.112


In [402]:
transport_par_region_2023_df.shape

(11, 6)

In [403]:
transport_par_region_2023_df["Region"].unique()

array(['Béni Mellal - Khénifra', 'Casablanca - Settat',
       'Drâa - Tafilalet', 'Fès - Meknès', 'Guelmim - Oued Noun',
       'Laâyoune - Sakia El Hamra', "L'Oriental", 'Marrakech - Safi',
       'Rabat - Salé - Kénitra', 'Souss - Massa',
       'Tanger - Tétouan - Al Hoceima'], dtype=object)

In [404]:
transport_par_region_2023_df.isnull().sum()

Region                             0
Annee                              0
Routes_Provinciales_Total_km       0
Routes_Provinciales_Revetues_km    0
Routes_Regionales_Total_km         0
Routes_Nationales_Total_km         0
dtype: int64

## Plage dataset

In [405]:
plage_par_region_df= pd.read_csv(path + '10_env_plages_par_region.csv')

plage_par_region_df.head()

,Region,Annee,Nb_Plages
0,Oriental,2023,16
1,Oriental,2022,16
2,Tanger - Tétouan- Al Hoceima,2023,74
3,Tanger - Tétouan- Al Hoceima,2022,72
4,Rabat - Salé - Kénitra,2023,18


In [406]:
plage_par_region_df.shape

(18, 3)

In [407]:
plage_par_region_df["Region"].unique()

array(['Oriental', 'Tanger - Tétouan- Al Hoceima',
       'Rabat - Salé - Kénitra', 'Casablanca - Settat',
       'Marrakech - Safi', 'Souss - Massa', 'Guelmim - Oued noun',
       'Lâayoune - SaKia El Hamra', 'Dakhla - Oued Eddahab'], dtype=object)

#### Vérification des duplicats

In [408]:
plage_par_region_df.duplicated(['Region','Annee']).sum()

0

#### Vérification des nulls

In [409]:
plage_par_region_df.isnull().sum()

Region       0
Annee        0
Nb_Plages    0
dtype: int64

# Water efficiency dataset

### Liste des fichiers qui concernent les regions

In [410]:
import os

path2 = "../data_raw/water_efficiency_dataset/"

files = [
    "Beni Mellal-Khénifra.csv",
    "Casablanca-Settat.csv",
    "Drâa-Tafilalet.csv",
    "Fès-Meknès.csv",
    "Guelmim-Oued Noun.csv",
    "Marrakech-Safi.csv",
    "Oriental.csv",
    "Rabat-Salé-Kénitra.csv",
    "Souss-Massa.csv",
    "Tanger-Tétouan-Al Hoceïma.csv"
]

### Fonction de transformation pour faire un groupement mensuel

In [411]:
import pandas as pd

def process_file(filepath):
    df = pd.read_csv(filepath)
    
    # nettoyage
    df = df.drop(columns=['country'], errors='ignore')
    
    # datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['Annee'] = df['timestamp'].dt.year
    df['Mois'] = df['timestamp'].dt.month
    
    # groupement mensuel
    df_mensuel = df.groupby(['city', 'Annee', 'Mois'], as_index=False).agg({
        # météo → moyenne
        'temperature': 'mean',
        'wind_speed': 'mean',
        'humidity': 'mean',
        'wetbulb_temperature': 'mean',
        
        # précipitation → somme
        'precipitation': 'sum',
        
        # énergie → somme
        'Other renewables (including geothermal and biomass) - TWh': 'sum',
        'Biofuels consumption - TWh': 'sum',
        'Solar consumption - TWh': 'sum',
        'Wind consumption - TWh': 'sum',
        'Hydro consumption - TWh': 'sum',
        'Nuclear consumption - TWh': 'sum',
        'Gas consumption - TWh': 'sum',
        'Coal consumption - TWh': 'sum',
        'Oil consumption - TWh': 'sum',
        'Total renewables - TWh': 'sum',
        'Total fossil fuels - TWh': 'sum',
        'Total energy - TWh': 'sum',
        'Low carbon - TWh': 'sum',
        'Other - TWh': 'sum',
        
        # WUE + leakages → moyenne
        'WUE_FixedApproachDirect(L/KWh)': 'mean',
        'WUE_FixedColdWaterDirect(L/KWh)': 'mean',
        'WUE_Indirect(L/KWh)': 'mean',
        'Leakages (%)': 'mean',
        
        # colonnes constantes
        'climate_region': 'first'
    })
    
    return df_mensuel

### Application sur tous les fichiers

In [412]:
dfs_water_efficiency = []

for file in files:
    filepath = os.path.join(path2, file)
    df_processed = process_file(filepath)
    dfs_water_efficiency.append(df_processed)

### Fusionner tous les résultats

In [413]:
df_all_regions_water_efficiency = pd.concat(dfs_water_efficiency, ignore_index=True)

In [414]:
df_all_regions_water_efficiency.shape

(130, 27)

### Vérification des nulls

In [415]:
df_all_regions_water_efficiency.isnull().sum()

city                                                         0
Annee                                                        0
Mois                                                         0
temperature                                                  0
wind_speed                                                   0
humidity                                                     0
wetbulb_temperature                                          0
precipitation                                                0
Other renewables (including geothermal and biomass) - TWh    0
Biofuels consumption - TWh                                   0
Solar consumption - TWh                                      0
Wind consumption - TWh                                       0
Hydro consumption - TWh                                      0
Nuclear consumption - TWh                                    0
Gas consumption - TWh                                        0
Coal consumption - TWh                                 

# Normalisation des noms des régions

In [416]:
df_all_regions_water_efficiency["city"].unique()

array(['Beni Mellal-Khénifra', 'Casablanca-Settat', 'Drâa-Tafilalet',
       'Fès-Meknès', 'Guelmim-Oued Noun', 'Marrakech-Safi', 'Oriental',
       'Rabat-Salé-Kénitra', 'Souss-Massa', 'Tanger-Tétouan-Al Hoceïma'],
      dtype=object)

In [417]:
population_par_region_df["Region"].unique()

array(['Tanger - Tétouan - Al Hoceima', "L'Oriental", 'Fès - Meknès',
       'Rabat - Salé - Kénitra', 'Béni Mellal - Khénifra',
       'Casablanca - Settat', 'Marrakech - Safi', 'Drâa - Tafilalet',
       'Souss - Massa', 'Guelmim - Oued Noun',
       'Laâyoune - Sakia El Hamra', 'Dakhla - Oued Ed-Dahab'],
      dtype=object)

In [418]:
energie_par_region_df["Region"].unique()

array(['Tanger - Tétouan - Al Hoceima', "L'Oriental", 'Fès - Meknès',
       'Rabat - Salé - Kénitra', 'Marrakech - Safi', 'Souss - Massa',
       'Guelmim - Oued Noun', 'Laâyoune - Sakia El Hamra',
       'Dakhla - Oued Ed-Dahab'], dtype=object)

In [419]:
eau_production_par_region_df["Region"].unique()

array(['Béni Mellal - Khénifra', 'Casablanca - Settat',
       'Drâa - Tafilalet', 'Fès - Meknès', "L'Oriental",
       'Marrakech - Safi', 'Rabat - Salé - Kénitra', 'Souss - Massa',
       'Tanger - Tétouan - Al Hoceima'], dtype=object)

In [420]:
pib_par_region_2022_df["Region"].unique()

array(['Tanger - Tétouan - Al Hoceima', "L'Oriental", 'Fès - Meknès',
       'Rabat - Salé - Kénitra', 'Béni Mellal - Khénifra',
       'Casablanca - Settat', 'Marrakech - Safi', 'Drâa - Tafilalet',
       'Souss - Massa', 'Guelmim - Oued Noun',
       'Laâyoune - Sakia El Hamra', 'Dakhla - Oued Ed-Dahab'],
      dtype=object)

In [421]:
transport_par_region_2023_df["Region"].unique()

array(['Béni Mellal - Khénifra', 'Casablanca - Settat',
       'Drâa - Tafilalet', 'Fès - Meknès', 'Guelmim - Oued Noun',
       'Laâyoune - Sakia El Hamra', "L'Oriental", 'Marrakech - Safi',
       'Rabat - Salé - Kénitra', 'Souss - Massa',
       'Tanger - Tétouan - Al Hoceima'], dtype=object)

In [422]:
plage_par_region_df["Region"].unique()

array(['Oriental', 'Tanger - Tétouan- Al Hoceima',
       'Rabat - Salé - Kénitra', 'Casablanca - Settat',
       'Marrakech - Safi', 'Souss - Massa', 'Guelmim - Oued noun',
       'Lâayoune - SaKia El Hamra', 'Dakhla - Oued Eddahab'], dtype=object)

### Définition d'un format standard

In [423]:
standard_regions = [
    'Tanger - Tétouan - Al Hoceima',
    "L'Oriental",
    'Fès - Meknès',
    'Rabat - Salé - Kénitra',
    'Béni Mellal - Khénifra',
    'Casablanca - Settat',
    'Marrakech - Safi',
    'Drâa - Tafilalet',
    'Souss - Massa',
    'Guelmim - Oued Noun',
    'Laâyoune - Sakia El Hamra',
    'Dakhla - Oued Ed-Dahab'
]

### Création d'un mapping

In [424]:
region_mapping = {
    # variantes sans accents ou avec tirets collés
    'Beni Mellal-Khénifra': 'Béni Mellal - Khénifra',
    'Béni Mellal-Khénifra': 'Béni Mellal - Khénifra',

    'Casablanca-Settat': 'Casablanca - Settat',
    
    'Drâa-Tafilalet': 'Drâa - Tafilalet',
    
    'Fès-Meknès': 'Fès - Meknès',
    
    'Guelmim-Oued Noun': 'Guelmim - Oued Noun',
    'Guelmim - Oued noun': 'Guelmim - Oued Noun',
    
    'Marrakech-Safi': 'Marrakech - Safi',
    
    'Oriental': "L'Oriental",
    
    'Rabat-Salé-Kénitra': 'Rabat - Salé - Kénitra',
    
    'Souss-Massa': 'Souss - Massa',
    
    'Tanger-Tétouan-Al Hoceïma': 'Tanger - Tétouan - Al Hoceima',
    'Tanger - Tétouan- Al Hoceima': 'Tanger - Tétouan - Al Hoceima',
    
    'Lâayoune - SaKia El Hamra': 'Laâyoune - Sakia El Hamra',
    
    'Dakhla - Oued Eddahab': 'Dakhla - Oued Ed-Dahab'
}

### Appliquer sur chaque dataset

In [425]:
df_all_regions_water_efficiency['city'] = df_all_regions_water_efficiency['city'].replace(region_mapping)

In [426]:
population_par_region_df['Region'] = population_par_region_df['Region'].replace(region_mapping)
energie_par_region_df['Region'] = energie_par_region_df['Region'].replace(region_mapping)
eau_production_par_region_df['Region'] = eau_production_par_region_df['Region'].replace(region_mapping)
pib_par_region_2022_df['Region'] = pib_par_region_2022_df['Region'].replace(region_mapping)
transport_par_region_2023_df['Region'] = transport_par_region_2023_df['Region'].replace(region_mapping)
plage_par_region_df['Region'] = plage_par_region_df['Region'].replace(region_mapping)

## Sauvegarde des datasets transformées

In [427]:
plage_par_region_df.to_csv("../data_final/plage_par_region.csv", index=False)

In [428]:
plage_par_region_df.columns

Index(['Region', 'Annee', 'Nb_Plages'], dtype='object')

In [429]:
transport_par_region_2023_df.to_csv("../data_final/transport_par_region_2023.csv", index=False)

In [430]:
transport_par_region_2023_df.columns

Index(['Region', 'Annee', 'Routes_Provinciales_Total_km',
       'Routes_Provinciales_Revetues_km', 'Routes_Regionales_Total_km',
       'Routes_Nationales_Total_km'],
      dtype='object')

In [431]:
pib_par_region_2022_df.to_csv("../data_final/pib_par_region_2022.csv", index=False)

In [432]:
pib_par_region_2022_df.columns

Index(['Region', 'Annee', 'A00_Agriculture', 'A05_Peche', 'B00_Extraction',
       'C00_Industrie_Manuf', 'DE0_Elec_Eau', 'F00_Construction',
       'G00_Commerce', 'H00_Transport', 'I00_Hebergement', 'J00_InfoComm',
       'K00_Finance', 'L68_Immobilier', 'MN0_Recherche', 'O84_AdminPublique',
       'PQ8_Education', 'RS0_Autres', 'IS_Pt_Impots_Nets', 'PIB_Regional_MDH'],
      dtype='object')

In [433]:
eau_production_par_region_df.to_csv("../data_final/eau_production_par_region.csv", index=False)

In [434]:
eau_production_par_region_df.columns

Index(['Region', 'Annee', 'Production_Eau_totale_m3'], dtype='object')

In [435]:
energie_par_region_df.to_csv("../data_final/energie_par_region.csv", index=False)

In [436]:
energie_par_region_df.columns

Index(['Region', 'Annee', 'Ventes_Electricite_Mm_KWh'], dtype='object')

In [437]:
population_par_region_df.to_csv("../data_final/population_par_region.csv", index=False)

In [438]:
population_par_region_df.columns

Index(['Region', 'Annee', 'Milieu', 'Population'], dtype='object')

In [439]:
df_all_regions_water_efficiency.to_csv("../data_final/all_regions_water_efficiency.csv", index=False)

In [440]:
df_all_regions_water_efficiency.columns

Index(['city', 'Annee', 'Mois', 'temperature', 'wind_speed', 'humidity',
       'wetbulb_temperature', 'precipitation',
       'Other renewables (including geothermal and biomass) - TWh',
       'Biofuels consumption - TWh', 'Solar consumption - TWh',
       'Wind consumption - TWh', 'Hydro consumption - TWh',
       'Nuclear consumption - TWh', 'Gas consumption - TWh',
       'Coal consumption - TWh', 'Oil consumption - TWh',
       'Total renewables - TWh', 'Total fossil fuels - TWh',
       'Total energy - TWh', 'Low carbon - TWh', 'Other - TWh',
       'WUE_FixedApproachDirect(L/KWh)', 'WUE_FixedColdWaterDirect(L/KWh)',
       'WUE_Indirect(L/KWh)', 'Leakages (%)', 'climate_region'],
      dtype='object')